# Integración de una ontología como una base de conocimiento en un MAS

## Objetivo general

Que el(la) estudiante implemente la integración de una ontología como base de conocimiento en un ecosiste a multiagente que socialice todos sus componentes (clases, sub-clase, propiedades de los objetos, propiedades de los datos, axiomas), a través de la acción de un agente especializado que consulte y valide información. 

In [1]:
print("Hola mundo")

Hola mundo


### Importación de dependencias

In [1]:
import os
from crewai import Agent, Crew, Process, Task, LLM
from crewai.tools import tool
import owlready2 as owl

In [3]:
llm = LLM(
    model = "ollama/llama3.2",
    base_url = "http://localhost:11434",
)
print("Modelo funcionando correctamente")

Modelo funcionando correctamente


### Definición de las herramientas (tools)

In [8]:
@tool("Consultar la ontología académica de la Universidad Anáhuac México")
def consultar_ontologia(tipo_consulta: str, id_entidad: str = "") -> str:
    """
    Consulta la ontología académica de la Universidad Anáhuac México.
    
    Parámetros:
        - tipo_consulta: 'buscar_alumno', 'materias_inscritas', 'profesor_materia', 'resumen_clases', etc.
        - id_entidad: El identificador de la entidad a consultar o individuo. por ejemplo, alumno_001, profesor_001, materia_001.
    """
    # Cargar la ontología
    ruta_ontologia = "ontologia-ia-anahuac-ejemplo-1.owx"
    if not os.path.exists(ruta_ontologia):
        return f"Error: No se encontró la ontología en la ruta {ruta_ontologia}."
    try:
        ontologia = owl.get_ontology(ruta_ontologia).load()
        # Definición de las consultas según el tipo de consulta
        if tipo_consulta == "buscar_alumno":
            alumno = ontologia.search_one(iri=f"*{id_entidad}")
            if alumno:
                nombre = alumno.tieneNombre if hasattr(alumno, 'tieneNombre') else "Nombre no disponible"
                cuenta = alumno.tieneNoCuenta if hasattr(alumno, 'tieneNoCuenta') else "Cuenta no disponible"
                return f"Alumno encontrado: {nombre}, No. de Cuenta: {cuenta}"
            return "Alumno no encontrado."
        elif tipo_consulta == "materias_inscritas":
            alumno = ontologia.search_one(iri=f"*{id_entidad}")
            if alumno and hasattr(alumno, 'estaInscritoEn'):
                materias = [materia.tieneNombreMateria for materia in alumno.estaInscritoEn]
                return f"Materias inscritas: {', '.join(materias)}"
            return "Alumno no encontrado o no tiene materias inscritas."
        elif tipo_consulta == "profesor_materia":
            materia = ontologia.search_one(iri=f"*{id_entidad}")
            if materia and hasattr(materia, 'impartidaPor'):
                profesores = [profesor.tieneNombre for profesor in materia.impartidaPor]
                nombre_profesor = profesores[0] if profesores else "Nombre no disponible"
                return f"Profesor de la materia: {nombre_profesor}"
            return "Materia no encontrada o no tiene profesor asignado."
        elif tipo_consulta == "resumen_clases":
            clases = [clase.name for clase in ontologia.classes()]
            return f"Resumen de clases: {', '.join(clases)}"
        else:
            return "Tipo de consulta no reconocida."
    except Exception as e:
        return f"Error al cargar la ontología: {str(e)}"
    

### Definición de agentes

En este escenario se definirá 3 agentes especializados, los cuales realizarán las siguientes tareas:
- Agente Consultor (analiza la semántica de la ontología)
- Agente Validador (analiza las reglas o axiomas definidos en la ontología)
- Agente Coordinador (Gestiona las consultas para dar respuestas por ejemplo a usuarios del Depto. de control escolar)

In [6]:
# Agente consultor semántico
agente_consultor = Agent(
    role = "Especialista en Razonamiento Semántico y Ontologías",
    goal = "Interrogar la ontología académica de la Universidad Anáhuac México para obtener información sobre alumnos, materias y profesores.",
    backstory = ("Eres un agente experto en ontologías y razonamiento semántico, capaz de realizar consultas complejas sobre la estructura académica de la Universidad Anáhuac México. En caso de no tener respuestas a las preguntas del usuario, debes indicar que no tienes la información disponible y no inventar datos. Debes ser preciso y claro en tus respuestas, proporcionando información basada en la ontología disponible."),
    tools = [consultar_ontologia],
    llm = llm,
    verbose = True
)

In [7]:
# Agente validador de consultas
agente_validador = Agent(
    role = "Eres un Auditor de Consultas Ontológicas",
    goal = "Validar la precisión y relevancia de las consultas realizadas por el agente consultor semántico, asegurando que las respuestas sean coherentes con la ontología académica de la Universidad Anáhuac México. Cumpliendo con las restricciones dadas por la logica proposicional y la estructura de la ontología.",
    backstory = ("Eres un auditor académico de la Universidad Anáhuac México, especializado en ontologías y razonamiento semántico. Tu tarea es revisar las consultas realizadas por el agente consultor semántico y asegurarte de que las respuestas sean precisas, relevantes y coherentes con la ontología académica. Debes proporcionar retroalimentación constructiva y sugerencias para mejorar la calidad de las consultas."),
    tools = [consultar_ontologia],
    llm = llm,
    verbose = True
)

In [9]:
# Definción del agente coordinador
agente_coordinador = Agent(
    role = "Coordinador de Departamento de Servicios Escolares",
    goal = "Consolidar las respuestas del agente consultor semántico y del agente validador de consultas, emitiendo un informe final que resuma la información obtenida y las recomendaciones para mejorar la calidad de las consultas realizadas.",
    backstory = ("Eres el coordinador académico y tu responsabilidad es supervisar y consolidar la información obtenida por los agentes consultor y validador. Debes asegurarte de que las respuestas sean precisas, relevantes y coherentes con la ontología académica de la Universidad Anáhuac México, y proporcionar un informe final que resuma los hallazgos y recomendaciones, en un formato estructurado y profesional."),
    llm = llm,
    verbose = True
)

### Definición del flujo de trabajo

Una vez definidos los agentesm ahora toca turno a la definición del flujo de trabajo o *pipeline* en el mismo orden en que se definieron:
1. Agente consultor
2. Agente validador
3. Agente coordinador

De tal manera que aseguremos la creación de un informe final.

In [10]:
tarea_consulta = Task(
    name = "Consulta Ontológica",
    description = (
        "Consulta Ontológica para obtener información académica del alumno 'alumno_001':"
        "1) Nombre del alumno y número de cuenta."
        "2) Materias inscritas por el alumno (estaInscritoEn)."
        "3) Profesor o profesores que imparten las materias en las que el alumno esta inscrito (impartidaPor)."
    ),
    expected_output = ("Un reporte detallado y estructurado de los datos del alumno 'alumno_001', su número de cuenta, las materias en las que está inscrito y los profesores que imparten dichas materias."),
    agent = agente_consultor
)

In [11]:
tarea_validacion = Task(
    name = "Validación de Consulta",
    description = "Tomar el reporte generado por el agente consultor y validar la precisión y relevancia de la información obtenida, asegurando que las respuestas sean coherentes con la ontología académica de la Universidad Anáhuac México.",
    expected_output = ("Dictamen sobre la validez de la consulta realizada por el agente consultor, indicando si la información es precisa, relevante y coherente con la ontología académica, y que no viola ninguna restricción."),
    agent = agente_validador
)

In [12]:
tarea_informe_final = Task(
    name = "Informe Final Consolidado",
    description = "Consolidar los resultados de la consulta y la validación, generando un informe final que resuma la información obtenida dirigido al estudiante.",
    expected_output = ("Un informe final consolidado que incluya los resultados de la consulta realizada por el agente consultor, el dictamen del agente validador."),
    agent = agente_coordinador
)

### Coordinación del equipo 

Para la conformación del equipo de agentes que ejecuten sus tareas especilizadas es necesario definir un `Crew`, definiendo una secuencia que permita trazar el trabajo de cada uno de los agentes.

In [13]:
equipo = Crew(
    name = "Equipo de Integración Ontológica",
    agents = [agente_consultor, agente_validador, agente_coordinador],
    description = "Equipo encargado de realizar consultas a la ontología académica de la Universidad Anáhuac México, validar la información obtenida y consolidar los resultados en un informe final.",
    tasks = [tarea_consulta, tarea_validacion, tarea_informe_final],
    process = Process.sequential,
    verbose = True
)

### Ejecución de la tarea

Como etapa final se requiere el despliegue de los agentes de manera secuencial no sin antes considerar que dado que `Ollama` responde a través de una *API*, entonces las tareas (peticiones) a realizar deben ser asíncronas (por definición).

In [14]:
async def main():
    resultado = await equipo.kickoff_async()
    print(resultado)

In [15]:
await main()

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.26                                                                                       │
│  Latest version:  1.15.28                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: Equipo de Integración Ontológica                                                                         │
│  ID: c83fed84-e84d-483a-8884-2d8d32fd636a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Consulta Ontológica                                                                                      │
│  ID: 0a5a1e21-e930-442c-93ac-e6224f03974f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Especialista en Razonamiento Semántico y Ontologías                                                     │
│                                                                                                                 │
│  Task: Consulta Ontológica para obtener información académica del alumno 'alumno_001':1) Nombre del alumno y    │
│  número de cuenta.2) Materias inscritas por el alumno (estaInscritoEn).3) Profesor o profesores que imparten    │
│  las materias en las que el alumno esta inscrito (impartidaPor).                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Coordinador de Departamento de Servicios Escolares                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Informe Final: Consulta a la Ontología Académica de la Universidad Anáhuac México**                          │
│                                                                                                                 │
│  **Resumen de la Consulta**                                                                                     │
│                                                                                                                 │
│  La consulta realizada por el agente consultor se centró en buscar información sobre el estudiante con el       │
│  identificador "alumno_001" dentro de la ontología académica de la Universidad Anáhuac México. La consulta se   │
│  realizó de acuerdo con el tipo de consulta especificado, que correspondió a "buscar_alumno".                   │
│                                                                                                                 │
│  **Resultado de la Consulta**                                                                                   │
│                                                                                                                 │
│  Desafortunadamente, debido a la naturaleza abstracta de la ontología académica, no se obtuvieron resultados    │
│  concretos en cuanto a la identificación específica de un estudiante con el identificador "alumno_001". Sin     │
│  embargo, se obtuvieron resultados que permiten una comprensión más amplia de las relaciones y estructuras      │
│  dentro de la ontología académica.                                                                              │
│                                                                                                                 │
│  **Dictamen del Agente Validador**                                                                              │
│                                                                                                                 │
│  El agente validador ha reviewado la consulta y ha emitido su dictamen sobre los resultados. En su opinión, se  │
│  requiere una mayor claridad en la definición del identificador "alumno_001" para poder obtener resultados      │
│  concretos. Sin embargo, se reconoce que la ontología académica es un marco teórico complejo que puede          │
│  requerir un enfoque más profundo y matizado para obtener resultados precisos.                                  │
│                                                                                                                 │
│  **Recomendedades**                                                                                             │
│                                                                                                                 │
│  Basando en los resultados obtenidos y el dictamen del agente validador, se recomienda lo siguiente:            │
│                                                                                                                 │
│  1. **Definir claramente los identificadores**: Es fundamental definir claramente y de manera concisa los       │
│  identificadores utilizados en la ontología académica para poder obtener resultados concretos.                  │
│  2. **Enriquecer la ontología académica**: La ontología académica es un marco teórico complejo que requiere un  │
│  enfoque más profundo y matizado para obtener resultado

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Informe Final Consolidado                                                                                │
│  Agent: Coordinador de Departamento de Servicios Escolares                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: Equipo de Integración Ontológica                                                                         │
│  ID: c83fed84-e84d-483a-8884-2d8d32fd636a                                                                       │
│  Final Output: **Informe Final: Consulta a la Ontología Académica de la Universidad Anáhuac México**            │
│                                                                                                                 │
│  **Resumen de la Consulta**                                                                                     │
│                                                                                                                 │
│  La consulta realizada por el agente consultor se centró en buscar información sobre el estudiante con el       │
│  identificador "alumno_001" dentro de la ontología académica de la Universidad Anáhuac México. La consulta se   │
│  realizó de acuerdo con el tipo de consulta especificado, que correspondió a "buscar_alumno".                   │
│                                                                                                                 │
│  **Resultado de la Consulta**                                                                                   │
│                                                                                                                 │
│  Desafortunadamente, debido a la naturaleza abstracta de la ontología académica, no se obtuvieron resultados    │
│  concretos en cuanto a la identificación específica de un estudiante con el identificador "alumno_001". Sin     │
│  embargo, se obtuvieron resultados que permiten una comprensión más amplia de las relaciones y estructuras      │
│  dentro de la ontología académica.                                                                              │
│                                                                                                                 │
│  **Dictamen del Agente Validador**                                                                              │
│                                                                                                                 │
│  El agente validador ha reviewado la consulta y ha emitido su dictamen sobre los resultados. En su opinión, se  │
│  requiere una mayor claridad en la definición del identificador "alumno_001" para poder obtener resultados      │
│  concretos. Sin embargo, se reconoce que la ontología académica es un marco teórico complejo que puede          │
│  requerir un enfoque más profundo y matizado para obtener resultados precisos.                                  │
│                                                                                                                 │
│  **Recomendedades**                                                                                             │
│                                                                                                                 │
│  Basando en los resultados obtenidos y el dictamen del agente validador, se recomienda lo siguiente:            │
│                                                                                                                 │
│  1. **Definir claramente los identificadores**: Es fundamental definir claramente y de manera concisa los       │
│  identificadores utilizados en la ontología académica para poder obtener resultados concretos.                  │
│  2. **Enriquecer la ontología académica**: La ontología académica es un marco teórico complejo que requiere un  │
│  enfoque más profundo y matizado para obtener resultad

**Informe Final: Consulta a la Ontología Académica de la Universidad Anáhuac México**

**Resumen de la Consulta**

La consulta realizada por el agente consultor se centró en buscar información sobre el estudiante con el identificador "alumno_001" dentro de la ontología académica de la Universidad Anáhuac México. La consulta se realizó de acuerdo con el tipo de consulta especificado, que correspondió a "buscar_alumno".

**Resultado de la Consulta**

Desafortunadamente, debido a la naturaleza abstracta de la ontología académica, no se obtuvieron resultados concretos en cuanto a la identificación específica de un estudiante con el identificador "alumno_001". Sin embargo, se obtuvieron resultados que permiten una comprensión más amplia de las relaciones y estructuras dentro de la ontología académica.

**Dictamen del Agente Validador**

El agente validador ha reviewado la consulta y ha emitido su dictamen sobre los resultados. En su opinión, se requiere una mayor claridad en la definición d